In [1]:
import os
import pandas as pd
import numpy as np

In [13]:
RAW_DATA_PATH = "../data/raw/no2_ccr_copy/"
stations = os.listdir("../data/raw/no2_ccr_copy")

In [14]:
ground_no2 = []

required = ['From Date', 'To Date', 'NO2', 'lat', 'lon']

for station in stations:
    st_path = os.path.join(RAW_DATA_PATH, station)

    try:
        df = pd.read_csv(
            st_path,
            encoding="utf-8-sig",
            usecols=required
        )

    except Exception as e:
        print(f"Error reading {station}: {e}")
        continue

    # skip duplicate file
    if station == "UP_Moradabad copy.csv":
        continue

    # fix incorrect lat, lon value in "WB_Kolkata_Victoria.csv"
    if station == "WB_Kolkata_Victoria.csv":
        df["lat"] = 22.544697
        df["lon"] = 88.342606

    df["From Date"] = pd.to_datetime(
        df["From Date"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    df["To Date"] = pd.to_datetime(
        df["To Date"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    df["NO2"] = pd.to_numeric(
        df["NO2"],
        errors="coerce"
    )

    df["lat"] = pd.to_numeric(
        df["lat"].astype(str).str.strip(),
        errors="coerce"
    )

    df["lon"] = pd.to_numeric(
        df["lon"].astype(str).str.strip(),
        errors="coerce"
    )

    df = df[
        (df["From Date"] >= "2018-01-01") &
        (df["From Date"] < "2020-01-01")
    ]

    df = df.dropna(subset=["NO2", "lat", "lon"])

    df["station"] = station[:-4]
    ground_no2.append(df)

ground_no2 = pd.concat(ground_no2, ignore_index=True)

print("Done concatenating!")
print(f"Rows: {len(ground_no2):,}")

Error reading .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte
Done concatenating!
Rows: 2,013,837


In [15]:
os.makedirs("../data/processed/no2_ground_station/", exist_ok=True)
ground_no2.to_csv("../data/processed/no2_ground_station/ground_no2_2018_2019.csv", index=False)